# Tuần 17: Cây Quyết Định (Decision Tree) - Bài Toán Phân Lớp Dữ Liệu

---

## Hôm nay chúng ta học gì?

Hãy tưởng tượng bạn đang đi xem bóng đá tại sân vận động. Sau trận đấu, ban tổ chức muốn biết: **"Khán giả có hài lòng với trải nghiệm hay không?"**

Thay vì hỏi từng người, họ có thể dùng **Cây Quyết Định (Decision Tree)** — một thuật toán Machine Learning giống như một **sơ đồ câu hỏi Yes/No** — để tự động dự đoán câu trả lời dựa trên các thông tin như: chất lượng ghế ngồi, thức ăn, thời tiết, v.v.

### Lộ trình bài học hôm nay:

| STT | Nội dung | Mô tả |
|-----|----------|-------|
| 1 | Tổng quan Machine Learning | Bức tranh toàn cảnh |
| 2 | Regression vs Classification | Hai bài toán cốt lõi |
| 3 | Cây Quyết Định là gì? | Cấu trúc và cách hoạt động |
| 4 | Chỉ số Gini | Cách cây "chọn" câu hỏi tốt nhất |
| 5 | Thực hành với dữ liệu thật | Xây dựng mô hình từ A-Z |
| 6 | Tinh chỉnh siêu tham số | Chống overfitting |
| 7 | Random Forest | "Nhiều cái đầu tốt hơn một" |

In [ ]:
# === Import tất cả thư viện cần dùng ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, classification_report

# Cấu hình biểu đồ đẹp
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Tắt cảnh báo không cần thiết
import warnings
warnings.filterwarnings('ignore')

print("Đã import xong tất cả thư viện! Sẵn sàng học bài.")

## 1. Tổng quan về Machine Learning (Học Máy)

**Machine Learning (Học Máy)** là một nhánh của Trí Tuệ Nhân Tạo (AI), giúp máy tính **tự học từ dữ liệu** mà không cần lập trình cụ thể cho từng nhiệm vụ.

### So sánh dễ hiểu:
- **Lập trình truyền thống:** Con người viết luật → Máy làm theo luật
- **Machine Learning:** Con người cho dữ liệu → Máy **tự tìm ra luật**

### 4 nhánh chính của Machine Learning:

| Nhánh | Ý nghĩa | Ví dụ đời thường |
|-------|---------|-----------------|
| **Supervised Learning** (Học có giám sát) | Học từ dữ liệu đã có nhãn/đáp án | Học sinh làm bài có đáp án sẵn |
| **Unsupervised Learning** (Học không giám sát) | Tìm cấu trúc ẩn trong dữ liệu chưa gán nhãn | Phân nhóm khách hàng theo hành vi mua sắm |
| **Deep Learning** (Học sâu) | Dùng mạng nơ-ron nhiều lớp cho dữ liệu phức tạp | Nhận diện khuôn mặt trong ảnh |
| **Reinforcement Learning** (Học tăng cường) | Học qua tương tác và phản hồi từ môi trường | Robot học chơi cờ qua hàng triệu ván |

Hôm nay chúng ta tập trung vào **Supervised Learning** — cụ thể là bài toán **Phân lớp (Classification)**.

## 2. Hai bài toán cốt lõi của Supervised Learning

Trong Supervised Learning, có **2 loại bài toán chính**, phân biệt dựa trên câu hỏi bạn muốn trả lời:

### Regression (Hồi quy) — Trả lời: "**Bao nhiêu?**"
- Đầu ra là một **con số liên tục**
- Ví dụ: Giá nhà là bao nhiêu? Nhiệt độ ngày mai? Doanh thu tháng tới?
- Tuần 16 chúng ta đã học Linear Regression!

### Classification (Phân lớp) — Trả lời: "**Loại nào?**"
- Đầu ra là một **nhãn/lớp** trong danh sách hữu hạn
- Ví dụ: Email này là spam hay không spam? Khách hàng hài lòng hay không? Bệnh nhân mắc bệnh gì?

### Mẹo nhớ nhanh:
> Nếu đáp án là **một con số** → Regression  
> Nếu đáp án là **một nhóm/loại** → Classification

**Hôm nay** chúng ta học **Classification** với thuật toán **Decision Tree** (Cây Quyết Định)!

## 3. Cây Quyết Định (Decision Tree) là gì?

### Ý tưởng đơn giản nhất

Hãy nghĩ về cách bạn quyết định **có đi xem phim tối nay hay không**:

```
          Thời tiết tốt?
          /          \
        Không         Có
        |              |
    Ở nhà 🏠      Có tiền không?
                   /          \
                 Không         Có
                 |              |
             Ở nhà 🏠     Đi xem phim 🎬
```

Đó chính là một **Cây Quyết Định**! Nó là một chuỗi các câu hỏi Yes/No dẫn đến quyết định cuối cùng.

### 3 thành phần chính:

| Thành phần | Ý nghĩa | Trong ví dụ trên |
|------------|---------|-----------------|
| **Root Node** (Nút gốc) | Câu hỏi đầu tiên, quan trọng nhất | "Thời tiết tốt?" |
| **Internal Node** (Nút trung gian) | Các câu hỏi tiếp theo | "Có tiền không?" |
| **Leaf Node** (Nút lá) | Kết quả/quyết định cuối cùng | "Ở nhà" hoặc "Đi xem phim" |

### Ưu và nhược điểm:

| Ưu điểm | Nhược điểm |
|----------|-----------|
| Dễ hiểu, dễ giải thích (như sơ đồ luồng) | Dễ bị **overfitting** (học thuộc lòng dữ liệu) |
| Không cần chuẩn hóa dữ liệu | Không ổn định — dữ liệu thay đổi nhỏ có thể tạo cây hoàn toàn khác |
| Tự động chọn đặc trưng quan trọng | Kém với biến liên tục |

## 4. Cây quyết định "chọn" câu hỏi bằng cách nào? — Chỉ số Gini

Vấn đề cốt lõi: Tại nút gốc, cây nên hỏi câu gì trước? "Thời tiết tốt?" hay "Có tiền?" hay "Có bạn đi cùng?"

Câu trả lời: Cây chọn câu hỏi nào giúp **phân chia dữ liệu "sạch" nhất** — tức là mỗi nhóm sau khi chia chỉ chứa một loại.

### Chỉ số Gini — Đo độ "lẫn lộn"

$$Gini = 1 - \sum_{i=1}^{n} p_i^2$$

Trong đó $p_i$ là tỷ lệ của lớp thứ $i$ trong nhóm.

**Hiểu đơn giản:**
- **Gini = 0** → Nhóm hoàn toàn "tinh khiết" (chỉ có 1 loại) — **Tốt nhất!**
- **Gini = 0.5** → Nhóm lẫn lộn tối đa (50/50) — **Tệ nhất!**

### So sánh với đời thường:
Hãy tưởng tượng bạn có một rổ trái cây:
- Rổ A: 10 táo, 0 cam → Gini = 0 (rất "sạch")
- Rổ B: 5 táo, 5 cam → Gini = 0.5 (rất "lộn xộn")
- Rổ C: 8 táo, 2 cam → Gini = 0.32 (khá "sạch")

In [ ]:
# === Minh họa Chỉ số Gini bằng code ===

def tinh_gini(so_lop_A, so_lop_B):
    """Tính chỉ số Gini cho một nhóm có 2 lớp"""
    tong = so_lop_A + so_lop_B
    if tong == 0:
        return 0
    p_A = so_lop_A / tong
    p_B = so_lop_B / tong
    gini = 1 - (p_A**2 + p_B**2)
    return gini

# Tính Gini cho 3 rổ trái cây
ro_A = tinh_gini(10, 0)   # 10 táo, 0 cam
ro_B = tinh_gini(5, 5)    # 5 táo, 5 cam  
ro_C = tinh_gini(8, 2)    # 8 táo, 2 cam

print("=== Chỉ số Gini cho mỗi rổ trái cây ===")
print(f"Rổ A (10 táo, 0 cam): Gini = {ro_A:.2f}  ← Hoàn toàn tinh khiết!")
print(f"Rổ B (5 táo, 5 cam):  Gini = {ro_B:.2f}  ← Lẫn lộn tối đa!")
print(f"Rổ C (8 táo, 2 cam):  Gini = {ro_C:.2f}  ← Khá sạch")

# Vẽ biểu đồ minh họa
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Rổ A
labels_A = ['Táo'] * 10
colors_A = ['#ff6b6b'] * 10
axes[0].bar(['Táo', 'Cam'], [10, 0], color=['#ff6b6b', '#ffa502'])
axes[0].set_title(f'Rổ A: Gini = {ro_A:.2f}\n(Tinh khiết!)', fontsize=13)
axes[0].set_ylabel('Số lượng')

# Rổ B
axes[1].bar(['Táo', 'Cam'], [5, 5], color=['#ff6b6b', '#ffa502'])
axes[1].set_title(f'Rổ B: Gini = {ro_B:.2f}\n(Lẫn lộn tối đa!)', fontsize=13)

# Rổ C
axes[2].bar(['Táo', 'Cam'], [8, 2], color=['#ff6b6b', '#ffa502'])
axes[2].set_title(f'Rổ C: Gini = {ro_C:.2f}\n(Khá sạch)', fontsize=13)

plt.suptitle('Chỉ số Gini càng THẤP → Nhóm càng "sạch" (tốt hơn)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Thực hành: Xây dựng Cây Quyết Định từ A-Z

### Bài toán thực tế
Chúng ta có dữ liệu khảo sát **khán giả xem bóng đá tại sân vận động** với ~127,000 khán giả. Mục tiêu: **Dự đoán khán giả Hài Lòng hay Không Hài Lòng** dựa trên các yếu tố như ghế ngồi, thức ăn, thời tiết, v.v.

### Quy trình Machine Learning tổng quát:
1. **Đọc & khám phá dữ liệu** → Hiểu dữ liệu trước khi làm gì
2. **Tiền xử lý** → Dọn dẹp dữ liệu (xử lý null, mã hóa text)
3. **Chia dữ liệu** → Train set (80%) + Test set (20%)
4. **Huấn luyện mô hình** → Cho cây học từ train set
5. **Đánh giá** → Kiểm tra trên test set

In [ ]:
# === Bước 1: Đọc và khám phá dữ liệu ===
df = pd.read_csv('data_decision_tree_bt.csv')

print("=== 5 dòng đầu tiên của dữ liệu ===")
print(df.head())
print(f"\n=== Kích thước dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột ===")
print(f"\n=== Các cột trong dữ liệu ===")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# === Khám phá biến mục tiêu: DoHaiLong ===
print("=== Phân bố biến mục tiêu (DoHaiLong) ===")
print(df['DoHaiLong'].value_counts())
print()

# Vẽ biểu đồ phân bố
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ cột
df['DoHaiLong'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Số lượng Hài Lòng vs Không Hài Lòng', fontsize=13)
axes[0].set_ylabel('Số lượng khán giả')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Biểu đồ tròn
df['DoHaiLong'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                     colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Tỷ lệ Hài Lòng vs Không Hài Lòng', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n→ Dữ liệu khá cân bằng giữa 2 nhóm — tốt cho việc huấn luyện mô hình!")

In [ ]:
# === Bước 2: Tiền xử lý dữ liệu ===

# 2a. Kiểm tra dữ liệu thiếu (null)
print("=== Kiểm tra dữ liệu thiếu ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])  # Chỉ hiển thị cột có null
print(f"\n→ Cột 'DoTrePhut' có {df['DoTrePhut'].isnull().sum()} giá trị thiếu")
print(f"  (chiếm {df['DoTrePhut'].isnull().sum()/len(df)*100:.2f}% dữ liệu)")

# Xóa các dòng có giá trị thiếu
df_clean = df.dropna()
print(f"\n→ Sau khi xóa null: {len(df_clean):,} dòng còn lại (mất {len(df)-len(df_clean)} dòng)")

# 2b. Mã hóa biến text thành số (One-Hot Encoding)
# Máy tính chỉ hiểu SỐ, nên phải chuyển text → số
print("\n=== Các cột dạng text cần mã hóa ===")
text_cols = df_clean.select_dtypes(include='object').columns.tolist()
for col in text_cols:
    print(f"  - {col}: {df_clean[col].unique()}")

# Dùng get_dummies để mã hóa
df_encoded = pd.get_dummies(df_clean, drop_first=True, dtype=int)

# Đổi tên cột cho dễ đọc
df_encoded = df_encoded.rename(columns={
    'DoHaiLong_Không Hài Lòng': 'DoHaiLong',
    'LoaiKhachHang_KH Trung Thành': 'LoaiKhachHang',
    'LoaiHinh_Đi Cùng Gia Đình/Bạn Bè': 'LoaiHinh'
})

print(f"\n→ Sau mã hóa: {df_encoded.shape[1]} cột (tất cả là số)")
print("  DoHaiLong: 0 = Hài Lòng, 1 = Không Hài Lòng")

In [ ]:
# === Bước 3: Chia dữ liệu thành Train và Test ===

# Tách biến mục tiêu (Y) và biến đầu vào (X)
Y = df_encoded['DoHaiLong']           # Cột cần dự đoán
X = df_encoded.drop(columns=['DoHaiLong', 'TT'])  # Bỏ cột mục tiêu và cột STT

# Chia 80% để huấn luyện, 20% để kiểm tra
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print("=== Chia dữ liệu Train/Test ===")
print(f"  Tổng dữ liệu:   {len(X):,} mẫu")
print(f"  Tập huấn luyện:  {len(X_train):,} mẫu (80%)")
print(f"  Tập kiểm tra:    {len(X_test):,} mẫu (20%)")
print(f"  Số đặc trưng:    {X.shape[1]} cột")
print(f"\n→ Giống như ôn thi: 80% bài tập để học, 20% đề thi để kiểm tra!")

In [ ]:
# === Bước 4: Huấn luyện Cây Quyết Định (chưa tinh chỉnh) ===

# Tạo mô hình cây quyết định (để mặc định, chưa giới hạn gì)
tree_v1 = DecisionTreeClassifier(random_state=42)

# Huấn luyện mô hình bằng dữ liệu train
tree_v1.fit(X_train, Y_train)

# Dự đoán trên tập test
tree_v1_pred = tree_v1.predict(X_test)

# === Bước 5: Đánh giá mô hình ===
print("=== Kết quả Cây Quyết Định v1 (mặc định, chưa tinh chỉnh) ===\n")

accuracy = accuracy_score(Y_test, tree_v1_pred)
f1 = f1_score(Y_test, tree_v1_pred)
recall = recall_score(Y_test, tree_v1_pred)
precision = precision_score(Y_test, tree_v1_pred)

print(f"  Accuracy  (Độ chính xác tổng): {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"  Precision (Đoán đúng khi nói KHL): {precision:.4f} ({precision*100:.1f}%)")
print(f"  Recall    (Tìm đúng bao nhiêu KHL): {recall:.4f} ({recall*100:.1f}%)")
print(f"  F1 Score  (Trung bình hài hòa):    {f1:.4f} ({f1*100:.1f}%)")

print(f"\n→ Mô hình đoán đúng {accuracy*100:.1f}% trường hợp — khá tốt cho lần đầu!")

### Giải thích các thước đo đánh giá

Giả sử mô hình dự đoán 100 khán giả "Không Hài Lòng":

| Thước đo | Ý nghĩa | Ví dụ dễ hiểu |
|----------|---------|---------------|
| **Accuracy** | Tổng số dự đoán đúng / Tổng số mẫu | "Trong 100 người, mô hình đoán đúng bao nhiêu?" |
| **Precision** | Trong số được dự đoán là KHL, bao nhiêu thực sự KHL? | "Trong 100 người bị gắn nhãn KHL, bao nhiêu thực sự KHL?" |
| **Recall** | Trong số thực sự KHL, mô hình tìm ra được bao nhiêu? | "Trong 100 người thực sự KHL, mô hình phát hiện bao nhiêu?" |
| **F1 Score** | Trung bình hài hòa của Precision và Recall | Cân bằng giữa 2 thước đo trên |

In [ ]:
# === Trực quan hóa Cây Quyết Định (3 tầng đầu) ===

plt.figure(figsize=(20, 10))
plot_tree(tree_v1, 
          filled=True,                                    # Tô màu theo lớp
          feature_names=X.columns,                        # Tên các đặc trưng
          class_names=['Hài Lòng', 'Không Hài Lòng'],    # Tên các lớp
          max_depth=3,                                    # Chỉ vẽ 3 tầng đầu
          rounded=True,                                   # Bo góc đẹp hơn
          fontsize=9)
plt.title('Cây Quyết Định (3 tầng đầu) — Đọc từ trên xuống dưới', fontsize=16)
plt.tight_layout()
plt.show()

print("Cách đọc cây:")
print("  - Mỗi ô là một NÚT, chứa câu hỏi (ví dụ: TraiNghiemDatVeOnline <= 1.5)")
print("  - Nếu ĐÚNG → đi sang TRÁI, nếu SAI → đi sang PHẢI")
print("  - 'gini': độ lẫn lộn tại nút đó")
print("  - 'samples': số mẫu dữ liệu tại nút")
print("  - 'class': lớp chiếm đa số tại nút")

In [ ]:
# === Thử dự đoán 10 khán giả đầu tiên trong tập test ===

X_test_10 = X_test.iloc[:10]
predictions_10 = tree_v1.predict(X_test_10)
actual_10 = Y_test.iloc[:10].values

print("=== So sánh Dự đoán vs Thực tế (10 khán giả đầu) ===\n")
print(f"{'STT':<5} {'Dự đoán':<20} {'Thực tế':<20} {'Đúng/Sai'}")
print("-" * 60)
for i in range(10):
    du_doan = "Không Hài Lòng" if predictions_10[i] == 1 else "Hài Lòng"
    thuc_te = "Không Hài Lòng" if actual_10[i] == 1 else "Hài Lòng"
    dung_sai = "ĐÚNG" if predictions_10[i] == actual_10[i] else "SAI"
    print(f"{i+1:<5} {du_doan:<20} {thuc_te:<20} {dung_sai}")

so_dung = sum(predictions_10 == actual_10)
print(f"\n→ Đoán đúng {so_dung}/10 khán giả!")

## 6. Tinh chỉnh siêu tham số — Chống Overfitting

### Overfitting là gì?

Hãy tưởng tượng một học sinh **học thuộc lòng** 100 câu hỏi trong sách. Khi gặp câu hỏi mới hơi khác một chút, học sinh đó không trả lời được. Đó chính là **overfitting** — mô hình học thuộc dữ liệu huấn luyện nhưng không tổng quát hóa được cho dữ liệu mới.

### 3 siêu tham số quan trọng để kiểm soát cây:

| Tham số | Ý nghĩa | Giá trị mặc định | Khuyến nghị |
|---------|---------|-------------------|-------------|
| **`max_depth`** | Độ sâu tối đa của cây | None (không giới hạn) | Bắt đầu với 5-10 |
| **`min_samples_split`** | Số mẫu tối thiểu để nút được chia tiếp | 2 | 10-50 |
| **`min_samples_leaf`** | Số mẫu tối thiểu ở mỗi nút lá | 1 | 5-20 |

### So sánh dễ hiểu:
- **`max_depth`** = Giới hạn "cây cao bao nhiêu" (thấp → đơn giản, cao → phức tạp)
- **`min_samples_split`** = "Cần bao nhiêu người mới được chia nhóm tiếp" (ít → chia quá chi tiết)
- **`min_samples_leaf`** = "Mỗi nhóm cuối phải có ít nhất bao nhiêu người" (ít → nhóm quá nhỏ, dễ sai)

In [ ]:
# === Thử cây quyết định với giới hạn siêu tham số ===

tree_v2 = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,            # Giới hạn cây sâu tối đa 5 tầng
    min_samples_split=10,   # Cần ít nhất 10 mẫu để chia tiếp
    min_samples_leaf=5      # Mỗi nút lá phải có ít nhất 5 mẫu
)
tree_v2.fit(X_train, Y_train)
tree_v2_pred = tree_v2.predict(X_test)

# Đánh giá
acc_v2 = accuracy_score(Y_test, tree_v2_pred)
f1_v2 = f1_score(Y_test, tree_v2_pred)

print("=== Cây Quyết Định v2 (có giới hạn siêu tham số) ===\n")
print(f"  Accuracy: {acc_v2:.4f} ({acc_v2*100:.1f}%)")
print(f"  F1 Score: {f1_v2:.4f} ({f1_v2*100:.1f}%)")
print(f"\n→ Accuracy giảm nhẹ so với v1, nhưng mô hình ĐƠN GIẢN hơn và ÍT OVERFITTING hơn!")
print("→ Đây là sự đánh đổi: đơn giản hơn ↔ mất chút độ chính xác")

### GridSearchCV — Tự động tìm bộ siêu tham số tốt nhất

Thay vì thử từng bộ tham số bằng tay, ta dùng **GridSearchCV** — nó sẽ tự động thử TẤT CẢ các tổ hợp và chọn bộ tốt nhất!

**Ý tưởng:** Giống như bạn thử nấu phở với nhiều lượng gia vị khác nhau, rồi chọn công thức ngon nhất.

In [ ]:
# === GridSearchCV: Tự động tìm bộ tham số tốt nhất ===

# Định nghĩa các giá trị muốn thử cho mỗi tham số
hyper_params = {
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20, 45],
    'min_samples_leaf': [2, 5, 10, 20, 45]
}

# Tính tổng số tổ hợp
total = len(hyper_params['max_depth']) * len(hyper_params['min_samples_split']) * len(hyper_params['min_samples_leaf'])
print(f"Tổng số tổ hợp cần thử: {total} x 5 fold CV = {total*5} lần huấn luyện")
print("Đang chạy GridSearchCV... (có thể mất 1-2 phút)\n")

# Các thước đo cần đánh giá
scoring = {'accuracy': 'accuracy', 'f1': 'f1', 'recall': 'recall', 'precision': 'precision'}

# Chạy GridSearchCV
tree_grid = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(
    estimator=tree_grid,
    param_grid=hyper_params,
    scoring=scoring,
    refit='accuracy',    # Chọn mô hình tốt nhất theo accuracy
    cv=5,                # Cross-validation 5 fold
    n_jobs=-1            # Dùng tất cả CPU cores
)
grid_search.fit(X_train, Y_train)

# Kết quả
print("=== Bộ tham số TỐT NHẤT tìm được ===")
print(f"  max_depth:         {grid_search.best_params_['max_depth']}")
print(f"  min_samples_split: {grid_search.best_params_['min_samples_split']}")
print(f"  min_samples_leaf:  {grid_search.best_params_['min_samples_leaf']}")
print(f"\n  Accuracy tốt nhất (cross-validation): {grid_search.best_score_:.4f}")

In [ ]:
# === Đánh giá mô hình tốt nhất từ GridSearch trên tập test ===

best_tree = grid_search.best_estimator_
best_pred = best_tree.predict(X_test)

acc_best = accuracy_score(Y_test, best_pred)
f1_best = f1_score(Y_test, best_pred)
recall_best = recall_score(Y_test, best_pred)
precision_best = precision_score(Y_test, best_pred)

print("=== Cây Quyết Định v3 (GridSearchCV - tốt nhất) ===\n")
print(f"  Accuracy:  {acc_best:.4f} ({acc_best*100:.1f}%)")
print(f"  Precision: {precision_best:.4f} ({precision_best*100:.1f}%)")
print(f"  Recall:    {recall_best:.4f} ({recall_best*100:.1f}%)")
print(f"  F1 Score:  {f1_best:.4f} ({f1_best*100:.1f}%)")

print("\n=== So sánh 3 phiên bản ===")
print(f"  v1 (mặc định):    Accuracy = {accuracy_score(Y_test, tree_v1_pred):.4f}")
print(f"  v2 (thủ công):    Accuracy = {acc_v2:.4f}")
print(f"  v3 (GridSearch):  Accuracy = {acc_best:.4f}  ← TỐT NHẤT!")

print("\n→ GridSearchCV giúp tìm bộ tham số tối ưu tự động!")

### Feature Importance — Tính thế nào?

Mỗi lần cây tách một nút, chỉ số Gini sẽ **giảm** (dữ liệu "sạch" hơn). Feature Importance của một đặc trưng = **tổng mức giảm Gini** tại tất cả các nút mà đặc trưng đó được dùng, có tính trọng số theo số mẫu đi qua nút.

$$\text{Importance}(feature) = \sum_{\text{các nút dùng feature}} \frac{n_{\text{nút}}}{n_{\text{tổng}}} \times \Delta Gini$$

**Tóm gọn 1 câu:** Feature nào giúp cây "chia nhóm sạch" nhiều nhất và thường xuyên nhất → quan trọng nhất.

> **Ví dụ:** Nếu câu hỏi "TraiNghiemDatVeOnline > 3?" giúp giảm Gini từ 0.50 xuống 0.25 → mức giảm = 0.25. Cộng dồn tất cả các nút dùng feature này → ra Importance. Cuối cùng sklearn chuẩn hóa để tổng = 1.

In [ ]:
# === Feature Importance — Yếu tố nào quan trọng nhất? ===

important_features = pd.Series(best_tree.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 7))
important_features.plot(kind='barh', color='#3498db')
plt.title('Mức độ quan trọng của từng đặc trưng (Feature Importance)', fontsize=14)
plt.xlabel('Mức độ quan trọng (Importance Score)')
plt.ylabel('Đặc trưng')
plt.tight_layout()
plt.show()

print("=== Top 5 yếu tố quan trọng nhất ===")
top5 = important_features.sort_values(ascending=False).head(5)
for i, (name, score) in enumerate(top5.items(), 1):
    print(f"  {i}. {name}: {score:.4f}")

print("\n→ Đây là những yếu tố ảnh hưởng NHIỀU NHẤT đến sự hài lòng của khán giả!")

## 7. Random Forest — "Nhiều cái đầu tốt hơn một"

### Vấn đề của cây đơn lẻ
Một cây quyết định đơn lẻ giống như hỏi ý kiến **một người** — có thể sai lệch. 

### Ý tưởng Random Forest
Random Forest giống như hỏi ý kiến **100 người khác nhau**, rồi lấy **đa số quyết định** — kết quả sẽ chính xác và ổn định hơn!

### 2 yếu tố "ngẫu nhiên" tạo sức mạnh:

| Yếu tố | Cách hoạt động | Tại sao hiệu quả? |
|--------|---------------|-------------------|
| **Bootstrap Sampling** | Mỗi cây học từ một **mẫu ngẫu nhiên** của dữ liệu (lấy có hoàn lại) | Mỗi cây nhìn thấy góc khác nhau của dữ liệu |
| **Feature Randomness** | Tại mỗi nút, chỉ xét một **tập con ngẫu nhiên** các đặc trưng | Ngăn chặn một đặc trưng mạnh lấn át tất cả |

### So sánh Decision Tree vs Random Forest:

| Tiêu chí | Decision Tree | Random Forest |
|----------|--------------|---------------|
| Cấu trúc | 1 cây đơn lẻ | Tập hợp nhiều cây |
| Độ chính xác | Trung bình, dễ overfitting | **Cao**, giảm overfitting |
| Tốc độ huấn luyện | Nhanh | Chậm hơn |
| Khả năng giải thích | Cao (White Box) | Thấp (Black Box) |
| Độ ổn định | Thấp | **Cao** |

In [ ]:
# === Huấn luyện Random Forest và so sánh ===

forest = RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1)
forest.fit(X_train, Y_train)
forest_pred = forest.predict(X_test)

acc_forest = accuracy_score(Y_test, forest_pred)
f1_forest = f1_score(Y_test, forest_pred)
recall_forest = recall_score(Y_test, forest_pred)
precision_forest = precision_score(Y_test, forest_pred)

print("=== Random Forest (100 cây) ===\n")
print(f"  Accuracy:  {acc_forest:.4f} ({acc_forest*100:.1f}%)")
print(f"  Precision: {precision_forest:.4f} ({precision_forest*100:.1f}%)")
print(f"  Recall:    {recall_forest:.4f} ({recall_forest*100:.1f}%)")
print(f"  F1 Score:  {f1_forest:.4f} ({f1_forest*100:.1f}%)")

# So sánh tổng thể
print("\n" + "="*60)
print("        SO SÁNH TỔNG THỂ CÁC MÔ HÌNH")
print("="*60)
print(f"  {'Mô hình':<30} {'Accuracy':<12} {'F1 Score'}")
print(f"  {'-'*50}")
print(f"  {'Decision Tree v1 (mặc định)':<30} {accuracy_score(Y_test, tree_v1_pred):.4f}       {f1_score(Y_test, tree_v1_pred):.4f}")
print(f"  {'Decision Tree v2 (thủ công)':<30} {acc_v2:.4f}       {f1_v2:.4f}")
print(f"  {'Decision Tree v3 (GridSearch)':<30} {acc_best:.4f}       {f1_best:.4f}")
print(f"  {'Random Forest (100 cây)':<30} {acc_forest:.4f}       {f1_forest:.4f}  ← BEST!")
print("="*60)

In [ ]:
# === Biểu đồ so sánh trực quan ===

models = ['Tree v1\n(mặc định)', 'Tree v2\n(thủ công)', 'Tree v3\n(GridSearch)', 'Random\nForest']
accuracies = [
    accuracy_score(Y_test, tree_v1_pred),
    acc_v2,
    acc_best,
    acc_forest
]
colors = ['#95a5a6', '#e67e22', '#2ecc71', '#3498db']

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=colors, edgecolor='black', linewidth=0.5)

# Thêm giá trị lên mỗi cột
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
             f'{acc:.1%}', ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.title('So sánh Accuracy giữa các mô hình', fontsize=15)
plt.ylabel('Accuracy')
plt.ylim(0.8, 0.95)
plt.tight_layout()
plt.show()

print("→ Random Forest cho kết quả TỐT NHẤT nhờ kết hợp sức mạnh của nhiều cây!")

## Tóm tắt & Ghi nhớ

### 7 điểm chính của bài học hôm nay:

| # | Nội dung | Ghi nhớ |
|---|---------|---------|
| 1 | **Machine Learning** có 4 nhánh | Supervised, Unsupervised, Deep Learning, Reinforcement |
| 2 | **Supervised Learning** có 2 bài toán | Regression ("Bao nhiêu?") vs Classification ("Loại nào?") |
| 3 | **Decision Tree** = sơ đồ câu hỏi Yes/No | Gồm: Nút gốc → Nút trung gian → Nút lá |
| 4 | **Chỉ số Gini** đo độ "tinh khiết" | Gini = 0 (tốt nhất), Gini = 0.5 (tệ nhất) |
| 5 | **Siêu tham số** chống overfitting | `max_depth`, `min_samples_split`, `min_samples_leaf` |
| 6 | **GridSearchCV** tự động tìm tham số tốt nhất | Thử tất cả tổ hợp, chọn tốt nhất |
| 7 | **Random Forest** = nhiều cây kết hợp | Chính xác hơn, ổn định hơn Decision Tree đơn lẻ |

### Quy trình tổng quát xây dựng mô hình phân lớp:

```
Đọc dữ liệu → Tiền xử lý (null, mã hóa) → Chia Train/Test 
→ Huấn luyện mô hình → Đánh giá → Tinh chỉnh siêu tham số → So sánh mô hình
```

### Liên hệ với các tuần trước:
- **Tuần 16 (Linear Regression):** Bài toán Regression — dự đoán con số liên tục
- **Tuần 17 (Decision Tree):** Bài toán Classification — dự đoán nhãn/lớp
- Cả hai đều là **Supervised Learning** nhưng khác nhau ở loại đầu ra!

### Tuần tiếp theo:
Tuần 18 sẽ học về **Pruning & Random Forest** chuyên sâu hơn — cách "cắt tỉa" cây để tối ưu!